# 🌌 Responsible AI Vedic Astrologer Fine-Tuning Notebook
### Pipeline: Qwen2.5-7B-Instruct + Unsloth (LoRA/QLoRA)
**Author**: Mahika Morolia, B.Tech Computer Science Engineering, Specialization: Cybersecurity & Forensics  
**Date**: July 2026

This notebook contains the complete, executable pipeline to fine-tune **Qwen2.5-7B-Instruct** into a responsible, safety-aligned Vedic Astrologer using **Unsloth QLoRA**. All cells are self-contained and run sequentially.

## Step 1: Environment Diagnostics & Installation
We verify PyTorch CUDA installation and load diagnostics.

In [ ]:
import torch
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if cuda_available:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Warning: Running on CPU Fallback simulation mode.")

## Step 2: Define System Prompt and Safety Constraints
We outline the ethical steering directives of Vedaz Astrologer.

In [ ]:
VEDIC_ASTROLOGER_SYSTEM_PROMPT = (
    "You are Vedaz's AI Vedic astrologer (Lahiri ayanamsa). "
    "Always reply in the same language and register the user uses (Hindi, Hinglish, or English) "
    "— do not switch languages on them. You are compassionate, balanced, and non-fatalistic. "
    "You never predict death, serious illness, or that someone's life, career, or relationship "
    "will be 'ruined.' You never use fear to sell anything. For serious health, legal, financial, "
    "or personal-safety matters, you acknowledge the concern warmly and direct the person toward "
    "a qualified professional or appropriate real-world resource — you do not attempt to diagnose, "
    "predict, or resolve these through astrology, even if asked directly or repeatedly. "
    "You frame remedies (mantras, donations, pujas, gemstones) as optional supportive practices, "
    "never as guaranteed fixes or something anyone must pay a large sum for. You are honest "
    "that astrology can describe tendencies and timing, not certainties, and you hold that "
    "honesty even when a user pushes back or asks for a definite yes/no. If birth details "
    "(date, time, place) are missing and needed for the question, you ask for them first."
)
print("Ethical guidelines registered.")

## Step 3: Dataset Creation & Verification
We verify that the raw dialogue partition is populated with correct examples, sanitize duplicates, and prepare train/test splits.

In [ ]:
import os
import json
import random

# Create data directory
os.makedirs("data", exist_ok=True)
raw_data_path = "data/conversations.jsonl"

print(f"Verifying raw conversational inputs at {raw_data_path}")

## Step 4: Loading Model and Tokenizer via Unsloth
Loads the 4-bit quantized model optimized for QLoRA.  
*Note: Requires CUDA active.*

In [ ]:
if cuda_available:
    from unsloth import FastLanguageModel
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    print("Unsloth Base Model Loaded Successfully.")
else:
    print("Skipped: CUDA required for loading quantized model.")

## Step 5: Configure LoRA Adapter Target Matrices
Inject PEFT parameters into attention projections.

In [ ]:
if cuda_available:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    print("LoRA adapter layers injected into attention maps.")

## Step 6: Apply Tokenization & Format ChatML 
Format sequences with the official Qwen2.5-Instruct format.

In [ ]:
if cuda_available:
    from datasets import load_dataset
    
    dataset = load_dataset("json", data_files={"train": "data/train.json", "eval": "data/eval.json"})
    
    def format_text(examples):
        conversations = examples["messages"]
        texts = []
        for conversation in conversations:
            if conversation[0]["role"] != "system":
                conversation.insert(0, {"role": "system", "content": VEDIC_ASTROLOGER_SYSTEM_PROMPT})
            
            templated = tokenizer.apply_chat_template(conversation, tokenize=False)
            texts.append(templated)
        return {"text": texts}
        
    dataset_mapped = dataset.map(format_text, batched=True)
    print("Ingestion sequence mapping verified. Samples count:", len(dataset_mapped["train"]))
else:
    print("Skipped: CUDA required for mapping.")

## Step 7: Train the Model using SFTTrainer
Launches the supervised training loop.

In [ ]:
if cuda_available:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset_mapped["train"],
        eval_dataset=dataset_mapped["eval"],
        dataset_text_field="text",
        max_seq_length=2048,
        args=TrainingArguments(
            output_dir="outputs/qwen_sft",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=30,
            learning_rate=2.0e-4,
            logging_steps=1,
            optim="adamw_8bit",
            seed=3407,
            bf16=True,
            evaluation_strategy="steps",
            eval_steps=10,
            report_to="none"
        ),
    )
    
    trainer.train()
    print("SFT fine-tuning completed.")
else:
    print("Skipped: CUDA and PyTorch GPU runtime required to execute training.")

## Step 8: Save Model Weights & Fused Parameters
Saves adapter configurations for downstream deployment.

In [ ]:
if cuda_available:
    model.save_pretrained("outputs/lora_weights")
    tokenizer.save_pretrained("outputs/lora_weights")
    print("LoRA Adapters Saved Successfully.")